In [1]:
#******************************************************
#*
#* Name:         nb-02-pandas-weather-data
#*     
#* Design Phase:
#*     Author:   John Miner
#*     Date:     04-01-2026
#*     Purpose:  Load delta table using pandas.
#* 
#******************************************************/

In [2]:
%%capture

#
# get lakehouse abfs path
#

def get_lakehouse_abfs_path(key = "/default"):
    
    # get mount point
    try:
        path = list(filter(lambda x: x["mountPoint"] == key, notebookutils.fs.mounts()))[0]["source"]
    except:
        path = ""

    # return results
    return path

# get the path
lakehouse_path = get_lakehouse_abfs_path()


In [3]:
#
#  1 - grab low temp data
#

import pandas as pd
df1 = pd.read_csv("/lakehouse/default/Files/weather/low_temps.csv")
display(df1.head(5))


In [4]:
#
#  2 - grab high temp data
#

import pandas as pd
df2 = pd.read_csv("/lakehouse/default/Files/weather/high_temps.csv")
display(df2.head(5))


In [5]:
#
#  3 - join two data frames
#

df3 = pd.merge(df1, df2, on='date')
display(df3.head(5))

In [6]:
#
#  4 - prepare for writing
#

df3.rename(columns={'date': 'obs_date'}, inplace=True)
df3.rename(columns={'temp_x': 'obs_low_temp'}, inplace=True)
df3.rename(columns={'temp_y': 'obs_high_temp'}, inplace=True)
display(df3.head(5))


In [7]:
#
# 5 - remove target table
#

# set variables
schema_name = "bronze"
table_name = "pandas_weather_data"
table_path = f"{lakehouse_path}/Tables/{schema_name}/{table_name}"

# only fails on missing dir
try:
    notebookutils.fs.rm(table_path, True)
except:
    pass

In [8]:

#
# 6 - write delta lake table (7 sec)
#

from deltalake import write_deltalake

# message user
print(f"create delta table located at {table_path}")

# over write table (must be abfs path)
write_deltalake(table_path, df3, mode='overwrite')


create delta table located at abfss://9d12d1b1-e32e-4bbe-9753-f0715d21ad49@onelake.dfs.fabric.microsoft.com/f880ba3e-79f9-4297-8b7f-4e39ca3d158b/Tables/bronze/pandas_weather_data


In [9]:
%%tsql -artifact lh_python_vs_spark -type Lakehouse

-- may take some time to refresh
-- select * from sys.objects where is_ms_shipped = 0

select top 5 * from bronze.pandas_weather_data;

ProgrammingError: ('ID - 18767ef5-c5cd-4010-abd8-e5bcd5882c9e | traceid = 69e79d5a-fbb6-4bd5-a045-36cfde2cdb60 42000', "[42000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Failed to complete the command because the underlying location does not exist. Underlying data description: table 'bronze.pandas_weather_data', file 'https://onelake.dfs.fabric.microsoft.com/9d12d1b1-e32e-4bbe-9753-f0715d21ad49/f880ba3e-79f9-4297-8b7f-4e39ca3d158b/Tables/bronze/pandas_weather_data/0-f8b301fb-fe1e-451c-989f-4bc3e3567fb8-0.parquet'. (24596) (SQLExecDirectW)")